In [1]:

# 1. Environment + paths
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
import hashlib, json, math, os, platform, re, sys, textwrap

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

STAGE_ID = "08"
STAGE_NAME = "08_MANUSCRIPT_FIGURES_AND_TABLES_vFINAL"
PROTOCOL_VERSION = "SWARABYANJAN-CROSSPLATFORM-v1.0.0"
REPRODUCTION_VERSION = "SWARABYANJAN-MANUSCRIPT-REPRO-vFINAL-1.0.0"

PRIMARY_BUDGET = 0.10
SECONDARY_BUDGETS = [0.05, 0.20]
MODELS = ["ebm", "tfidf_lr", "banglabert", "xlmr"]
SEEDS = [42, 137, 2718, 31415, 65537]
CUES = [
    "curiosity_information_withholding",
    "direct_address_imperative",
    "exaggeration_superlative",
    "interrogative_structure",
    "punctuation_intensity",
    "sensational_emotional_loading",
    "specificity_numerals",
    "vague_deictic_reference",
]
PLATFORMS = ["news", "facebook", "youtube", "twitter"]

OUT_ROOT = Path(os.environ.get("SWARABYANJAN_STAGE08_OUT", "/kaggle/working/08_manuscript_outputs"))
if not Path("/kaggle/working").exists():
    OUT_ROOT = Path("/mnt/data/08_manuscript_outputs")

FIG_MAIN = OUT_ROOT / "figures" / "main"
FIG_SUPP = OUT_ROOT / "figures" / "supplementary"
TAB_MAIN = OUT_ROOT / "tables" / "main"
TAB_SUPP = OUT_ROOT / "tables" / "supplementary"
SRC_DATA = OUT_ROOT / "figure_source_data"
for d in [OUT_ROOT, FIG_MAIN, FIG_SUPP, TAB_MAIN, TAB_SUPP, SRC_DATA]:
    d.mkdir(parents=True, exist_ok=True)

SEARCH_ROOTS = []
for p in [
    os.environ.get("SWARABYANJAN_STAGE05_ROOT"),
    os.environ.get("SWARABYANJAN_STAGE06_ROOT"),
    os.environ.get("SWARABYANJAN_STAGE07_ROOT"),
    "/kaggle/input", "/kaggle/working", "/mnt/data", str(Path.cwd()),
]:
    if p:
        q = Path(p)
        if q.exists() and q not in SEARCH_ROOTS:
            SEARCH_ROOTS.append(q)

EXPECTED = {
    "stage05": {
        "stage_id": "05",
        "canonical_stage05": True,
        "canonical_version": "SWARABYANJAN-STAGE05-CANONICAL-vFINAL-1.0.1",
        "protocol_version": PROTOCOL_VERSION,
        "manifest_sha256": "7246bc3a49a5f8d8472005cea9f6062d118431cd67d5dd2c4fd8f2ee31f52c27",
        "checksum_sha256": "7171a0e581d8fb4bde2a630f4de631d74962954ee3890b8123d78a720823450a",
        "status_sha256": "005399d829bb66870810faa3f140757a7ee3d82eac4bae394fbbe79e701db8e6",
    },
    "stage06": {
        "stage_id": "06",
        "protocol_version": PROTOCOL_VERSION,
        "manifest_sha256": "5a2962014445c723379b07fcf02e601644818c36e6c15fb7ee9478ca34711285",
        "checksum_sha256": "5f8d0f7fc9cde5e0f159bf4c5f4ca8387441675407b4c43ab87a8063afac3beb",
        "status_sha256": "769f33b4771f3ec55700384ae6db7e9e2feb115d23717b27cbd9c17e4983538d",
        "h4_sha256": "299fc2e1394c80e1a090ac14d632e0f502b5b779c459e47dcb4e96130b621462",
    },
    "stage07": {
        "stage_id": "07",
        "protocol_version": PROTOCOL_VERSION,
        "manifest_sha256": "fa8651ff00a3306dfd48ed55d21df3ab2eef2fbd365db33657f5c2413822e714",
        "checksum_sha256": "86c8aeb1c4c0062303b44d397fece5908c315ff8860fca59afc624bfdde39f8d",
        "status_sha256": "566f063f1bef265c6d377565ee6a22ccc1cbbbaa083f7bad7a89423ec6222280",
        "bridge_sha256": "90ae9bc50b42640d3ca03c4e8697ea2b910e3c53de56e4b3c1df0a158f002deb",
    },
}

REQUIRED_STAGE05 = [
    "H1_RESULTS.json",
    "H2_RESULTS.json",
    "SENSITIVITY_RESULTS.json",
    "H3_RESULTS.json",
    "H3_SPARSITY_REPORT.csv",
    "H3_CUE_PLATFORM_CONTRASTS_CORRECTED.csv",
    "H3_BAITBUSTER_REPLICATION_CORRECTED.csv",
    "H3_PERTURBATION_RESULTS.json",
    "STATISTICAL_SUMMARY.json",
    "STAGE05_ARTIFACT_MANIFEST.json",
    "STAGE05_SHA256SUMS.txt",
    "STAGE_STATUS.json",
]
REQUIRED_STAGE06 = [
    "H4_RESULTS.json",
    "FIXED_BUDGET_TABLES.csv",
    "RISK_COVERAGE_DATA.csv",
    "FINAL_STATISTICAL_SUMMARY.json",
    "REPRODUCIBILITY_MANIFEST.json",
    "STAGE06_SHA256SUMS.txt",
    "UPSTREAM_PROVENANCE_GATE.json",
    "STAGE_STATUS.json",
]
REQUIRED_STAGE07 = [
    "BRIDGE_32_CELL_POINT_TABLE.csv",
    "BRIDGE_POINT_ASSOCIATIONS.json",
    "H3_H4_BRIDGE_RESULTS.json",
    "LEAVE_ONE_PLATFORM_OUT_POINT_TABLE.csv",
    "LEAVE_ONE_PLATFORM_OUT_ROBUSTNESS.json",
    "REPRODUCIBILITY_MANIFEST.json",
    "STAGE07_SHA256SUMS.txt",
    "UPSTREAM_PROVENANCE_GATE.json",
    "STAGE_STATUS.json",
]

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def json_builtin(x):
    if isinstance(x, np.bool_):
        return bool(x)
    if isinstance(x, np.integer):
        return int(x)
    if isinstance(x, np.floating):
        if not np.isfinite(x):
            return None
        return float(x)
    if isinstance(x, np.ndarray):
        return [json_builtin(v) for v in x.tolist()]
    if isinstance(x, Path):
        return str(x)
    if isinstance(x, dict):
        return {str(k): json_builtin(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [json_builtin(v) for v in x]
    return x

def write_json(path, obj):
    Path(path).write_text(
        json.dumps(json_builtin(obj), indent=2, sort_keys=True, ensure_ascii=False, allow_nan=False) + "\n",
        encoding="utf-8",
    )

def write_text(path, text):
    Path(path).write_text(str(text), encoding="utf-8")

def parse_sha256sums(path):
    out = {}
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        parts = line.split(None, 1)
        if len(parts) != 2:
            raise RuntimeError(f"Malformed checksum line in {path}: {line}")
        digest, rel = parts
        rel = rel.lstrip("*").strip().replace("\\", "/")
        if not re.fullmatch(r"[0-9a-fA-F]{64}", digest):
            raise RuntimeError(f"Malformed SHA-256 digest in {path}: {line}")
        if rel in out:
            raise RuntimeError(f"Duplicate checksum path in {path}: {rel}")
        out[rel] = digest.lower()
    return out

def verify_manifest_payload(manifest, label):
    if "manifest_payload_sha256" not in manifest:
        raise RuntimeError(f"{label}: manifest payload seal missing")
    recorded = manifest["manifest_payload_sha256"]
    tmp = dict(manifest)
    tmp.pop("manifest_payload_sha256", None)
    observed = hashlib.sha256(
        json.dumps(json_builtin(tmp), sort_keys=True, separators=(",", ":"), ensure_ascii=False).encode("utf-8")
    ).hexdigest()
    if observed != recorded:
        raise RuntimeError(f"{label}: manifest payload seal mismatch")
    return observed

def verify_checksum_contract(root, checksum_name, label):
    sums = parse_sha256sums(root / checksum_name)
    for rel, expected in sums.items():
        p = root / rel
        if not p.is_file():
            raise RuntimeError(f"{label}: checksum-contract artifact missing: {rel}")
        observed = sha256_file(p)
        if observed != expected:
            raise RuntimeError(f"{label}: checksum mismatch for {rel}")
    return sums

def discover_exact_root(manifest_name, expected_manifest_sha256, required_files):
    matches = []
    seen = set()
    for root in SEARCH_ROOTS:
        candidates = [root / manifest_name] if root.is_dir() else []
        if root.is_dir():
            try:
                candidates += list(root.rglob(manifest_name))
            except Exception:
                pass
        for p in candidates:
            if not p.is_file():
                continue
            d = p.parent.resolve()
            if d in seen:
                continue
            seen.add(d)
            try:
                if sha256_file(p) != expected_manifest_sha256:
                    continue
            except Exception:
                continue
            if all((d / f).is_file() for f in required_files):
                matches.append(d)
    matches = sorted(set(matches), key=str)
    return matches

def present_basenames(names):
    found = {}
    for name in names:
        paths = []
        for root in SEARCH_ROOTS:
            if not root.is_dir():
                continue
            try:
                paths.extend(str(p) for p in root.rglob(name) if p.is_file())
            except Exception:
                pass
        found[name] = sorted(set(paths))
    return found

FIGURE_REGISTRY = []
TABLE_REGISTRY = []
SOURCE_REGISTRY = []
OMISSIONS = []
VALIDATION = {}


In [2]:

# 2. Canonical provenance validation
def hard_stop_missing(stage_label, required_files, detail=None):
    present = present_basenames(required_files)
    # A same-named file outside the exact canonical stage root does not satisfy
    # the input contract. When exact-root discovery fails, every required file
    # in that canonical root is treated as unavailable for reproduction.
    missing = list(required_files)
    payload = {
        "schema_version": 1,
        "stage_id": STAGE_ID,
        "status": "HARD_STOP_MISSING_CANONICAL_INPUT",
        "timestamp_utc": utc_now(),
        "blocked_stage": stage_label,
        "missing_artifacts": missing,
        "required_artifacts": required_files,
        "present_same_name_paths": present,
        "searched_roots": [str(x) for x in SEARCH_ROOTS],
        "rule": "No reconstruction from notebook outputs, screenshots, memory, or manually typed values.",
        "detail": detail,
        "figures_generated": [],
        "tables_generated": [],
    }
    write_json(OUT_ROOT / "FIGURE_REPRODUCTION_AUDIT.json", payload)
    write_json(OUT_ROOT / "MISSING_INPUTS.json", payload)
    write_json(OUT_ROOT / "STAGE08_STATUS.json", {
        "status": "HARD_STOP",
        "reason": f"Missing canonical {stage_label} inputs",
        "missing_artifacts": missing,
        "timestamp_utc": payload["timestamp_utc"],
    })
    # Create a minimal reproducibility manifest/checksum contract for the blocked run.
    current = ["FIGURE_REPRODUCTION_AUDIT.json", "MISSING_INPUTS.json", "STAGE08_STATUS.json"]
    manifest = {
        "schema_version": 1,
        "stage_id": STAGE_ID,
        "reproduction_version": REPRODUCTION_VERSION,
        "status": "BLOCKED",
        "canonical_inputs_complete": False,
        "blocked_stage": stage_label,
        "missing_artifacts": missing,
        "artifacts": {name: sha256_file(OUT_ROOT / name) for name in current},
    }
    write_json(OUT_ROOT / "MANIFEST.json", manifest)
    lines = []
    for name in sorted(current + ["MANIFEST.json"]):
        lines.append(f"{sha256_file(OUT_ROOT / name)}  {name}")
    write_text(OUT_ROOT / "STAGE08_SHA256SUMS.txt", "\n".join(lines) + "\n")
    raise RuntimeError(
        f"Stage 08 HARD STOP: canonical {stage_label} input contract is incomplete. "
        f"Missing: {missing}"
    )

S5_MATCHES = discover_exact_root(
    "STAGE05_ARTIFACT_MANIFEST.json",
    EXPECTED["stage05"]["manifest_sha256"],
    REQUIRED_STAGE05,
)
if not S5_MATCHES:
    hard_stop_missing(
        "Stage-05",
        REQUIRED_STAGE05,
        detail="The canonical Stage-05 result directory is required before any figure or table is drawn.",
    )
S5 = S5_MATCHES[0]

S6_MATCHES = discover_exact_root(
    "REPRODUCIBILITY_MANIFEST.json",
    EXPECTED["stage06"]["manifest_sha256"],
    REQUIRED_STAGE06,
)
if not S6_MATCHES:
    hard_stop_missing("Stage-06", REQUIRED_STAGE06)
S6 = S6_MATCHES[0]

S7_MATCHES = discover_exact_root(
    "REPRODUCIBILITY_MANIFEST.json",
    EXPECTED["stage07"]["manifest_sha256"],
    REQUIRED_STAGE07,
)
if not S7_MATCHES:
    hard_stop_missing("Stage-07", REQUIRED_STAGE07)
S7 = S7_MATCHES[0]

# Stage 05 exact identity.
if sha256_file(S5 / "STAGE05_SHA256SUMS.txt") != EXPECTED["stage05"]["checksum_sha256"]:
    raise RuntimeError("Stage-05 checksum-file identity mismatch")
if sha256_file(S5 / "STAGE_STATUS.json") != EXPECTED["stage05"]["status_sha256"]:
    raise RuntimeError("Stage-05 status-file identity mismatch")
S5_MAN = json.loads((S5 / "STAGE05_ARTIFACT_MANIFEST.json").read_text(encoding="utf-8"))
verify_manifest_payload(S5_MAN, "Stage-05")
for k in ["stage_id", "canonical_stage05", "canonical_version", "protocol_version", "artifacts"]:
    if k not in S5_MAN:
        raise RuntimeError(f"Stage-05 required manifest key missing: {k}")
if S5_MAN["stage_id"] != EXPECTED["stage05"]["stage_id"]:
    raise RuntimeError("Stage-05 stage_id mismatch")
if S5_MAN["canonical_stage05"] is not True:
    raise RuntimeError("Stage-05 canonical_stage05 != True")
if S5_MAN["canonical_version"] != EXPECTED["stage05"]["canonical_version"]:
    raise RuntimeError("Stage-05 canonical version mismatch")
if S5_MAN["protocol_version"] != PROTOCOL_VERSION:
    raise RuntimeError("Stage-05 protocol version mismatch")
S5_STATUS = json.loads((S5 / "STAGE_STATUS.json").read_text(encoding="utf-8"))
if S5_STATUS.get("status") != "PASS":
    raise RuntimeError("Stage-05 status is not PASS")
S5_SUMS = verify_checksum_contract(S5, "STAGE05_SHA256SUMS.txt", "Stage-05")
for f in REQUIRED_STAGE05:
    if f in {
        "STAGE05_ARTIFACT_MANIFEST.json",
        "STAGE05_SHA256SUMS.txt",
        "STAGE_STATUS.json",
    }:
        continue

    if f not in S5_MAN["artifacts"] or f not in S5_SUMS:
        raise RuntimeError(f"Stage-05 provenance omits required figure input: {f}")

    observed = sha256_file(S5 / f)
    if S5_MAN["artifacts"][f] != observed or S5_SUMS[f] != observed:
        raise RuntimeError(f"Stage-05 figure input identity mismatch: {f}")

# Stage 06 exact identity and upstream Stage-05 binding.
if sha256_file(S6 / "STAGE06_SHA256SUMS.txt") != EXPECTED["stage06"]["checksum_sha256"]:
    raise RuntimeError("Stage-06 checksum-file identity mismatch")
if sha256_file(S6 / "STAGE_STATUS.json") != EXPECTED["stage06"]["status_sha256"]:
    raise RuntimeError("Stage-06 status-file identity mismatch")
if sha256_file(S6 / "H4_RESULTS.json") != EXPECTED["stage06"]["h4_sha256"]:
    raise RuntimeError("Stage-06 H4 identity mismatch")
S6_MAN = json.loads((S6 / "REPRODUCIBILITY_MANIFEST.json").read_text(encoding="utf-8"))
verify_manifest_payload(S6_MAN, "Stage-06")
S6_SUMS = verify_checksum_contract(S6, "STAGE06_SHA256SUMS.txt", "Stage-06")
S6_STATUS = json.loads((S6 / "STAGE_STATUS.json").read_text(encoding="utf-8"))
if S6_STATUS.get("status") != "PASS":
    raise RuntimeError("Stage-06 status is not PASS")
if S6_MAN.get("stage_id") != "06" or S6_MAN.get("protocol_version") != PROTOCOL_VERSION:
    raise RuntimeError("Stage-06 identity/version mismatch")
up6 = S6_MAN.get("upstream")
if not isinstance(up6, dict):
    raise RuntimeError("Stage-06 upstream provenance missing")
if up6.get("mode") != "DIRECT_CANONICAL_STAGE05_OUTPUT":
    raise RuntimeError("Stage-06 does not point to canonical Stage-05")
if up6.get("stage05_manifest_sha256") != EXPECTED["stage05"]["manifest_sha256"]:
    raise RuntimeError("Stage-06 Stage-05 manifest identity mismatch")
if up6.get("stage05_checksum_sha256") != EXPECTED["stage05"]["checksum_sha256"]:
    raise RuntimeError("Stage-06 Stage-05 checksum identity mismatch")
if up6.get("stage05_status_sha256") != EXPECTED["stage05"]["status_sha256"]:
    raise RuntimeError("Stage-06 Stage-05 status identity mismatch")

# Stage 07 exact identity and upstream chain.
if sha256_file(S7 / "STAGE07_SHA256SUMS.txt") != EXPECTED["stage07"]["checksum_sha256"]:
    raise RuntimeError("Stage-07 checksum-file identity mismatch")
if sha256_file(S7 / "STAGE_STATUS.json") != EXPECTED["stage07"]["status_sha256"]:
    raise RuntimeError("Stage-07 status-file identity mismatch")
if sha256_file(S7 / "H3_H4_BRIDGE_RESULTS.json") != EXPECTED["stage07"]["bridge_sha256"]:
    raise RuntimeError("Stage-07 bridge identity mismatch")
S7_MAN = json.loads((S7 / "REPRODUCIBILITY_MANIFEST.json").read_text(encoding="utf-8"))
verify_manifest_payload(S7_MAN, "Stage-07")
S7_SUMS = verify_checksum_contract(S7, "STAGE07_SHA256SUMS.txt", "Stage-07")
S7_STATUS = json.loads((S7 / "STAGE_STATUS.json").read_text(encoding="utf-8"))
if S7_STATUS.get("status") != "PASS":
    raise RuntimeError("Stage-07 status is not PASS")
if S7_MAN.get("stage_id") != "07" or S7_MAN.get("protocol_version") != PROTOCOL_VERSION:
    raise RuntimeError("Stage-07 identity/version mismatch")
up7 = S7_MAN.get("upstream")
if not isinstance(up7, dict) or up7.get("mode") != "EXACT_CANONICAL_STAGE05_TO_STAGE06_CHAIN":
    raise RuntimeError("Stage-07 canonical-chain provenance missing")
for key, expected in [
    ("stage05_manifest_sha256", EXPECTED["stage05"]["manifest_sha256"]),
    ("stage05_checksum_sha256", EXPECTED["stage05"]["checksum_sha256"]),
    ("stage05_status_sha256", EXPECTED["stage05"]["status_sha256"]),
    ("stage06_manifest_sha256", EXPECTED["stage06"]["manifest_sha256"]),
    ("stage06_checksum_sha256", EXPECTED["stage06"]["checksum_sha256"]),
    ("stage06_status_sha256", EXPECTED["stage06"]["status_sha256"]),
    ("stage06_h4_sha256", EXPECTED["stage06"]["h4_sha256"]),
]:
    if key not in up7 or up7[key] != expected:
        raise RuntimeError(f"Stage-07 upstream identity mismatch: {key}")

PROVENANCE_REPORT = {
    "status": "PASS",
    "stage05_root": str(S5),
    "stage06_root": str(S6),
    "stage07_root": str(S7),
    "stage05_manifest_sha256": EXPECTED["stage05"]["manifest_sha256"],
    "stage06_manifest_sha256": EXPECTED["stage06"]["manifest_sha256"],
    "stage07_manifest_sha256": EXPECTED["stage07"]["manifest_sha256"],
    "canonical_chain": "Stage 05 FINAL → Stage 06 FINAL → Stage 07 FINAL → Stage 08 reproduction",
}
write_json(OUT_ROOT / "PROVENANCE_REPORT.json", PROVENANCE_REPORT)
VALIDATION["canonical_provenance"] = True
print("Canonical Stage-05 → Stage-06 → Stage-07 provenance: PASS")


Canonical Stage-05 → Stage-06 → Stage-07 provenance: PASS


In [3]:

# 3. Input loading
H1 = json.loads((S5 / "H1_RESULTS.json").read_text(encoding="utf-8"))
H2 = json.loads((S5 / "H2_RESULTS.json").read_text(encoding="utf-8"))
SENS = json.loads((S5 / "SENSITIVITY_RESULTS.json").read_text(encoding="utf-8"))
H3 = json.loads((S5 / "H3_RESULTS.json").read_text(encoding="utf-8"))
H3P = json.loads((S5 / "H3_PERTURBATION_RESULTS.json").read_text(encoding="utf-8"))
S5_STAT = json.loads((S5 / "STATISTICAL_SUMMARY.json").read_text(encoding="utf-8"))
H3_SPARSE = pd.read_csv(S5 / "H3_SPARSITY_REPORT.csv")
H3_BC = pd.read_csv(S5 / "H3_CUE_PLATFORM_CONTRASTS_CORRECTED.csv")
H3_BB = pd.read_csv(S5 / "H3_BAITBUSTER_REPLICATION_CORRECTED.csv")

H4 = json.loads((S6 / "H4_RESULTS.json").read_text(encoding="utf-8"))
S6_STAT = json.loads((S6 / "FINAL_STATISTICAL_SUMMARY.json").read_text(encoding="utf-8"))
H4_FIXED = pd.read_csv(S6 / "FIXED_BUDGET_TABLES.csv")
RISK_COVERAGE = pd.read_csv(S6 / "RISK_COVERAGE_DATA.csv")

BRIDGE_CELLS = pd.read_csv(S7 / "BRIDGE_32_CELL_POINT_TABLE.csv")
BRIDGE_ASSOC = json.loads((S7 / "BRIDGE_POINT_ASSOCIATIONS.json").read_text(encoding="utf-8"))
BRIDGE = json.loads((S7 / "H3_H4_BRIDGE_RESULTS.json").read_text(encoding="utf-8"))
LOO_POINTS = pd.read_csv(S7 / "LEAVE_ONE_PLATFORM_OUT_POINT_TABLE.csv")
LOO = json.loads((S7 / "LEAVE_ONE_PLATFORM_OUT_ROBUSTNESS.json").read_text(encoding="utf-8"))

INPUT_HASHES = {}
for label, root, files in [
    ("stage05", S5, REQUIRED_STAGE05),
    ("stage06", S6, REQUIRED_STAGE06),
    ("stage07", S7, REQUIRED_STAGE07),
]:
    INPUT_HASHES[label] = {f: sha256_file(root / f) for f in files}
write_json(OUT_ROOT / "INPUT_HASHES.json", INPUT_HASHES)
print("Canonical result artifacts loaded.")


Canonical result artifacts loaded.


In [4]:

# 4. Canonical schema validation
def require(cond, message):
    if not bool(cond):
        raise RuntimeError(message)

require(H1.get("schema_version") == 2, "Unexpected H1 schema")
require(H2.get("schema_version") == 2, "Unexpected H2 schema")
require(len(H1.get("metric_rows", [])) == 120, "H1 metric_rows must contain 120 domain×model×seed records")
require(len(H1.get("confirmatory_assessment_rows", [])) == 100, "H1 confirmatory rows must contain 100 target settings")
require(len(H2.get("confirmatory_assessment_rows", [])) == 100, "H2 confirmatory rows must contain 100 target settings")
require(len(H3_BC) == 32, "H3 BanglaClick table must contain exactly 32 cells")
require(len(H3_BB) == 8, "H3 BaitBuster replication table must contain exactly 8 cue families")
require(len(H3P.get("rows", [])) == 150, "H3-P must contain exactly 150 frozen contrasts")
require(set(H3_BC["cue"]) == set(CUES), "H3 cue family set mismatch")
require(set(H3_BC["target"]) == {f"banglaclick:{p}" for p in PLATFORMS}, "H3 platform set mismatch")
require(H3_BC[["cue", "target"]].drop_duplicates().shape[0] == 32, "H3 duplicate cue×platform cells")
require(int((H3_BC["inferential_status"] != "ESTIMABLE").sum()) == int(H3.get("nonestimable_banglaclick_contrast_count")), "H3 NE count mismatch")
require(H3.get("global_banglaclick_outcome_wald", {}).get("status") == "NON_ESTIMABLE", "H3 global status changed")

require(H4.get("budgets") == [0.05, 0.1, 0.2], "H4 budgets changed")
require(float(H4.get("primary_budget")) == PRIMARY_BUDGET, "H4 primary budget changed")
require(int(H4.get("bootstrap_replicates")) == 5000, "H4 bootstrap replicate count changed")
require(len(H4.get("fixed_budget_point_estimates", [])) == 360, "H4 point-estimate universe must be 360 rows")
require(len(H4.get("bootstrap_rows", [])) == 300, "H4 inferential universe must be 300 rows")
require(sorted(H4_FIXED["budget"].unique().tolist()) == [0.05, 0.1, 0.2], "FIXED_BUDGET_TABLES budgets changed")

require(len(BRIDGE_CELLS) == 32, "Stage-07 bridge point table must preserve 32 cells")
require(BRIDGE_CELLS[["cue", "platform"]].drop_duplicates().shape[0] == 32, "Bridge duplicate cells")
require(set(BRIDGE_CELLS["cue"]) == set(CUES), "Bridge cue family set mismatch")
require(set(BRIDGE_CELLS["platform"]) == set(PLATFORMS), "Bridge platform set mismatch")
require(int(BRIDGE.get("cells")) == 32, "Bridge result cell count changed")
require(int(BRIDGE.get("bootstrap_replicates")) == 5000, "Bridge bootstrap replicate count changed")
require(BRIDGE.get("exploratory_secondary") is True and BRIDGE.get("causal") is False, "Bridge scientific role changed")
require(len(BRIDGE.get("pairs", {})) == 4, "Bridge must contain exactly four locked association pairs")
require(len(LOO_POINTS) == 16, "LOO point table must contain 16 pair×omitted-platform rows")
require(LOO.get("analysis_role") == "SECONDARY_ROBUSTNESS_ONLY", "LOO scientific role changed")
require(LOO.get("primary_bridge_modified") is False, "LOO must not modify primary bridge")

# Cross-check Stage-05 H3 point quantities against Stage-07 bridge |Δ| quantities.
h3_lookup = {
    (r["cue"], str(r["target"]).split(":", 1)[1]): r
    for r in H3.get("banglaclick_cue_platform_contrasts", [])
}
require(len(h3_lookup) == 32, "H3 JSON contrast universe mismatch")
for row in BRIDGE_CELLS.to_dict("records"):
    key = (row["cue"], row["platform"])
    r = h3_lookup[key]
    require(np.isclose(float(row["abs_delta_rd"]), abs(float(r["delta_rd"])), rtol=0, atol=1e-12), f"Bridge |ΔRD| parity failure: {key}")
    require(np.isclose(float(row["abs_delta_nll_gap"]), abs(float(r["ebm_delta_nll_gap_mean_5seeds"])), rtol=0, atol=1e-12), f"Bridge |ΔNLL-gap| parity failure: {key}")

SCHEMA_REPORT = {
    "status": "PASS",
    "h1_metric_rows": len(H1["metric_rows"]),
    "h1_target_settings": len(H1["confirmatory_assessment_rows"]),
    "h2_target_settings": len(H2["confirmatory_assessment_rows"]),
    "h3_banglaclick_cells": len(H3_BC),
    "h3_nonestimable_cells": int((H3_BC["inferential_status"] != "ESTIMABLE").sum()),
    "h3p_contrasts": len(H3P["rows"]),
    "h4_point_rows": len(H4["fixed_budget_point_estimates"]),
    "h4_inference_rows": len(H4["bootstrap_rows"]),
    "bridge_cells": len(BRIDGE_CELLS),
    "bridge_pairs": len(BRIDGE["pairs"]),
    "loo_rows": len(LOO_POINTS),
}
write_json(OUT_ROOT / "SCHEMA_VALIDATION.json", SCHEMA_REPORT)
VALIDATION["canonical_schema"] = True
print("Canonical schema validation: PASS")


Canonical schema validation: PASS


In [5]:
# 5. Deterministic helper functions
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

matplotlib.rcParams.update({
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 7,
    "figure.titlesize": 11,
    "savefig.bbox": "tight",
})

MODEL_MARKERS = {"ebm": "o", "tfidf_lr": "s", "banglabert": "^", "xlmr": "D"}
MODEL_LABELS = {"ebm": "EBM", "tfidf_lr": "TF-IDF LR", "banglabert": "BanglaBERT", "xlmr": "XLM-R"}
MODEL_OFFSETS = {"ebm": -0.24, "tfidf_lr": -0.08, "banglabert": 0.08, "xlmr": 0.24}
PLATFORM_MARKERS = {"news": "o", "facebook": "s", "youtube": "^", "twitter": "D"}
PLATFORM_OFFSETS = {"news": -0.18, "facebook": -0.06, "youtube": 0.06, "twitter": 0.18}
SEED_LINESTYLES = {
    42: "-",
    137: "--",
    2718: ":",
    31415: "-.",
    65537: (0, (3, 1, 1, 1)),
}

MODEL_LEGEND_HANDLES = [
    Line2D([0], [0], marker=MODEL_MARKERS[m], linestyle="None", color="black",
           markerfacecolor="white", markeredgecolor="black", markersize=5,
           label=MODEL_LABELS[m])
    for m in MODELS
]
PLATFORM_LEGEND_HANDLES = [
    Line2D([0], [0], marker=PLATFORM_MARKERS[p], linestyle="None", color="black",
           markerfacecolor="white", markeredgecolor="black", markersize=5,
           label=p.title())
    for p in PLATFORMS
]
STATUS_LEGEND_HANDLES = [
    Line2D([0], [0], marker="o", linestyle="None", color="black", markerfacecolor="black", markersize=5, label="Canonical p < 0.05"),
    Line2D([0], [0], marker="o", linestyle="None", color="black", markerfacecolor="white", markersize=5, label="NS"),
    Line2D([0], [0], marker="x", linestyle="None", color="black", markersize=5, label="NE"),
]


def domain_label(x):
    mapping = {
        "source_test": "Source TEST",
        "banglaclick:news": "News",
        "banglaclick:facebook": "Facebook",
        "banglaclick:youtube": "YouTube",
        "banglaclick:twitter": "Twitter",
        "baitbuster:youtube_human": "BaitBuster",
    }
    return mapping.get(x, x)


def cue_label(x):
    return x.replace("_", " ")


def export_figure(fig, fig_id, title, main=True):
    outdir = FIG_MAIN if main else FIG_SUPP
    paths = {}
    for ext in ["pdf", "svg", "png"]:
        p = outdir / f"{fig_id}.{ext}"
        kwargs = {"dpi": 450} if ext == "png" else {}
        fig.savefig(p, **kwargs)
        paths[ext] = str(p.relative_to(OUT_ROOT))
    FIGURE_REGISTRY.append({"figure_id": fig_id, "title": title, "main": bool(main), "files": paths})
    plt.close(fig)


def register_source_data(fig_id, panel_id, df, source_artifacts, variables, transformation,
                         estimability_field=None, covers_figure_ids=None):
    data_name = f"{fig_id}_{panel_id}_source.csv".replace("/", "_")
    meta_name = f"{fig_id}_{panel_id}_source.json".replace("/", "_")
    data_path = SRC_DATA / data_name
    meta_path = SRC_DATA / meta_name
    out = df.copy()
    out.to_csv(data_path, index=False)
    covered = list(covers_figure_ids) if covers_figure_ids is not None else [fig_id]
    meta = {
        "figure_id": fig_id,
        "covers_figure_ids": covered,
        "panel_id": panel_id,
        "source_artifact": source_artifacts,
        "exact_variables_used": list(variables),
        "deterministic_transformation": transformation,
        "estimability_field": estimability_field,
        "rows": int(len(out)),
        "source_csv": data_name,
        "source_csv_sha256": sha256_file(data_path),
    }
    write_json(meta_path, meta)
    SOURCE_REGISTRY.append(meta)
    return data_path


def save_table(df, table_id, title, main=True):
    outdir = TAB_MAIN if main else TAB_SUPP
    p = outdir / f"{table_id}.csv"
    df.to_csv(p, index=False)
    TABLE_REGISTRY.append({"table_id": table_id, "title": title, "main": bool(main), "file": str(p.relative_to(OUT_ROOT)), "rows": int(len(df))})
    return p


def ci_parts(ci):
    if ci is None or (isinstance(ci, float) and np.isnan(ci)):
        return (np.nan, np.nan)
    if isinstance(ci, str):
        try:
            ci = json.loads(ci)
        except Exception:
            return (np.nan, np.nan)
    if isinstance(ci, (list, tuple)) and len(ci) == 2:
        return float(ci[0]), float(ci[1])
    return (np.nan, np.nan)


def endpoint_estimable(status_value):
    if isinstance(status_value, (bool, np.bool_)):
        return bool(status_value)
    return status_value == "ESTIMABLE"


def plot_interval_points(ax, rows, x_order, y_field, ci_field, status_field, title, ylabel, model_field="model"):
    x_lookup = {x: i for i, x in enumerate(x_order)}
    for r in rows:
        domain = r["domain"]
        model = r[model_field]
        x = x_lookup[domain] + MODEL_OFFSETS.get(model, 0)
        y = r.get(y_field)
        estimable = endpoint_estimable(r.get(status_field, "ESTIMABLE"))
        if y is None or not np.isfinite(float(y)):
            ax.plot(x, 0, marker="x", linestyle="None", color="black", markersize=4)
            continue
        y = float(y)
        if estimable:
            lo, hi = ci_parts(r.get(ci_field))
            if np.isfinite(lo) and np.isfinite(hi):
                ax.errorbar(x, y, yerr=[[y-lo], [hi-y]], fmt=MODEL_MARKERS.get(model, "o"),
                            color="black", markerfacecolor="white", markeredgecolor="black",
                            markersize=3.5, capsize=1.5, linewidth=0.7)
            else:
                ax.plot(x, y, marker=MODEL_MARKERS.get(model, "o"), linestyle="None",
                        color="black", markerfacecolor="white", markeredgecolor="black", markersize=3.5)
        else:
            ax.plot(x, y, marker="x", linestyle="None", color="black", markersize=4)
    ax.axhline(0, color="black", linewidth=0.7)
    ax.set_xticks(range(len(x_order)), [domain_label(x) for x in x_order], rotation=30, ha="right")
    ax.set_title(title)
    ax.set_ylabel(ylabel)


def flatten_metric_rows(metric_rows):
    flat = []
    for r in metric_rows:
        for cal_name in ["uncalibrated", "calibrated"]:
            m = r.get(cal_name, {})
            flat.append({
                "domain": r["domain"], "model": r["model"], "seed": r["seed"],
                "metric_state": cal_name,
                "auroc": m.get("auroc"), "nll": m.get("nll"),
                "balanced_brier": m.get("balanced_brier"),
                "calibration_intercept": m.get("calibration_intercept"),
                "calibration_slope": m.get("calibration_slope"),
            })
    return pd.DataFrame(flat)


def significance_label(status, p):
    if status != "ESTIMABLE":
        return "NE"
    if p is None or not np.isfinite(float(p)):
        return "NE"
    return "SIG" if float(p) < 0.05 else "NS"


def format_h3_display(df):
    out = df.copy()
    if "inferential_status" not in out.columns:
        return out
    ne = out["inferential_status"].ne("ESTIMABLE")
    for col in ["delta_rd_ci95", "delta_rd_p", "ebm_delta_nll_gap_ci95", "ebm_delta_nll_gap_p"]:
        if col in out.columns:
            out[col] = out[col].astype(object)
            out.loc[ne, col] = "NE"
    return out


def format_h4_display(df):
    out = df.copy()
    for status_col, ci_col, p_col in [
        ("lift_estimability_status", "lift_ci95", "lift_p"),
        ("error_capture_estimability_status", "error_capture_ci95", "error_capture_p"),
        ("aurc_estimability_status", "aurc_ci95", "aurc_p"),
    ]:
        if status_col in out.columns:
            ne = out[status_col].ne("ESTIMABLE")
            for col in [ci_col, p_col]:
                if col in out.columns:
                    out[col] = out[col].astype(object)
                    out.loc[ne, col] = "NE"
    return out


FIGURE_ISSUES = []


In [6]:
# 6. Figure 1 — Source-locked study design / evidence hierarchy
h3_global_status = H3.get("global_banglaclick_outcome_wald", {}).get("status")
bridge_statuses = sorted({
    v["bootstrap"]["platform_fe_slope"]["estimability_status"]
    for v in BRIDGE["pairs"].values()
})
nodes = [
    ("Frozen external data", "provenance", "pipeline"),
    ("Scientific decontamination", "data integrity", "pipeline"),
    ("Source-locked training", "model development", "pipeline"),
    ("DEV-only calibration", "calibration", "pipeline"),
    ("Real-platform transfer evaluation", "external evaluation", "pipeline"),
    ("H1 / H2", "supporting hypotheses", "supporting"),
    ("H3 cue transport", f"primary; global {h3_global_status}", "primary"),
    ("H3-P controlled perturbation", "controlled model-sensitivity evidence; non-causal", "controlled"),
    ("H4 fixed-budget utility", "operational consequence", "operational"),
    ("H3→H4 exploratory bridge", f"exploratory/non-causal; {', '.join(bridge_statuses)}", "exploratory"),
]
role_style = {
    "pipeline": dict(facecolor="white", edgecolor="black", linestyle="-", linewidth=0.8),
    "supporting": dict(facecolor="0.97", edgecolor="black", linestyle="--", linewidth=1.0),
    "primary": dict(facecolor="0.90", edgecolor="black", linestyle="-", linewidth=1.4),
    "controlled": dict(facecolor="white", edgecolor="black", linestyle="-.", linewidth=1.0),
    "operational": dict(facecolor="0.94", edgecolor="black", linestyle="-", linewidth=1.1),
    "exploratory": dict(facecolor="white", edgecolor="black", linestyle=":", linewidth=1.1),
}
fig, ax = plt.subplots(figsize=(7.4, 9.6))
ax.set_axis_off()

# Reserve the lower fifth of the canvas for the evidence-class legend and
# non-causal sequence note. Scientific node order/status data remain unchanged.
ys = np.linspace(0.95, 0.24, len(nodes))
for i, ((label, role, evidence_class), y) in enumerate(zip(nodes, ys)):
    st = role_style[evidence_class]
    box = FancyBboxPatch((0.16, y-0.032), 0.68, 0.064, boxstyle="round,pad=0.01", fill=True, **st)
    ax.add_patch(box)
    ax.text(0.50, y+0.009, label, ha="center", va="center", fontweight="bold")

    # Display-only manuscript wording for the exploratory bridge. The canonical
    # machine status stored in `nodes` and exported source data is untouched.
    display_role = role
    if evidence_class == "exploratory":
        display_role = "exploratory/non-causal; inference non-estimable under the locked joint bootstrap"
    ax.text(0.50, y-0.015, display_role, ha="center", va="center", fontsize=6.5)

    if i < len(nodes)-1:
        ax.annotate("", xy=(0.50, ys[i+1]+0.035), xytext=(0.50, y-0.035),
                    arrowprops=dict(arrowstyle="->", color="black", linewidth=0.8))
legend_handles = [
    Patch(facecolor="0.97", edgecolor="black", linestyle="--", label="Supporting"),
    Patch(facecolor="0.90", edgecolor="black", linestyle="-", label="Primary"),
    Patch(facecolor="white", edgecolor="black", linestyle="-.", label="Controlled corroboration"),
    Patch(facecolor="0.94", edgecolor="black", linestyle="-", label="Operational consequence"),
    Patch(facecolor="white", edgecolor="black", linestyle=":", label="Exploratory / non-causal"),
]
ax.legend(handles=legend_handles, loc="lower center", bbox_to_anchor=(0.5, 0.085),
          ncol=3, frameon=False, fontsize=7)
ax.text(0.50, 0.018, "Arrows denote analysis sequence, not causal effects.",
        fontsize=7, ha="center", va="bottom")
ax.set_title("Figure 1. Source-locked study design and evidence hierarchy", pad=10)
export_figure(fig, "FIG01_STUDY_DESIGN", "Source-locked study design and evidence hierarchy", main=True)
fig1_source = pd.DataFrame([
    {"order": i+1, "node": a, "role_or_status": b, "evidence_class": c}
    for i,(a,b,c) in enumerate(nodes)
])
register_source_data("FIG01_STUDY_DESIGN", "pipeline", fig1_source,
                     ["Stage-05/06/07 canonical manifests and final statuses"],
                     ["node", "role_or_status", "evidence_class"],
                     "Direct representation of the frozen analysis sequence and canonical inferential roles; line/fill styles encode predefined evidence classes only.")


PosixPath('/kaggle/working/08_manuscript_outputs/figure_source_data/FIG01_STUDY_DESIGN_pipeline_source.csv')

In [7]:
# 7. Figure 2 — Cross-platform discrimination + probability reliability
metric_df = pd.DataFrame(H1["metric_rows"])
h1_df = pd.DataFrame(H1["confirmatory_assessment_rows"])
h2_df = pd.DataFrame(H2["confirmatory_assessment_rows"])
boot_df = pd.DataFrame(S5_STAT["h1_h2_bootstrap"])

targets = ["banglaclick:news", "banglaclick:facebook", "banglaclick:youtube", "banglaclick:twitter", "baitbuster:youtube_human"]
all_domains = ["source_test"] + targets

# Panel A source/target calibrated AUROC: exact canonical point estimates only.
auroc_rows = []
for r in H1["metric_rows"]:
    auroc_rows.append({
        "domain": r["domain"], "model": r["model"], "seed": r["seed"],
        "calibrated_auroc": r["calibrated"]["auroc"],
    })
auroc_plot = pd.DataFrame(auroc_rows)

# Merge canonical bootstrap intervals/statuses into H1/H2 effect rows.
keys = ["domain", "model", "seed"]
bcols = keys + [
    "auroc_drop_ci95",
    "nll_relative_change_ci95",
    "balanced_brier_relative_change_ci95",
    "h2_inferential_status", "h2_target_minus_source_gain_ci95",
]
bsub = boot_df[bcols].copy()
h1m = h1_df.merge(bsub, on=keys, how="left", validate="one_to_one")
h2m = h2_df.merge(bsub[keys + ["h2_inferential_status", "h2_target_minus_source_gain_ci95"]], on=keys, how="left", validate="one_to_one")

fig, axes = plt.subplots(2, 2, figsize=(12, 8.5))
ax = axes[0,0]
x_lookup = {d:i for i,d in enumerate(all_domains)}
for r in auroc_plot.to_dict("records"):
    x = x_lookup[r["domain"]] + MODEL_OFFSETS.get(r["model"], 0)
    ax.plot(x, r["calibrated_auroc"], marker=MODEL_MARKERS.get(r["model"], "o"), linestyle="None",
            color="black", markerfacecolor="white", markeredgecolor="black", markersize=3.5)
ax.set_xticks(range(len(all_domains)), [domain_label(x) for x in all_domains], rotation=30, ha="right")
ax.set_ylabel("Calibrated AUROC")
ax.set_title("A. Canonical calibrated discrimination (model×seed points)")

plot_interval_points(
    axes[0,1], h1m.to_dict("records"), targets,
    "nll_relative_change", "nll_relative_change_ci95", "nll_relative_change_estimable",
    "B. H1 relative NLL change", "(Target − source) / source"
)

plot_interval_points(
    axes[1,0], h1m.to_dict("records"), targets,
    "balanced_brier_relative_change", "balanced_brier_relative_change_ci95", "balanced_brier_relative_change_estimable",
    "C. H1 relative balanced-Brier change", "(Target − source) / source"
)
# Preserve all 20 BaitBuster point estimates/x markers while replacing overlapping per-point NE text with one compact annotation.
bb_brier = h1m[h1m["domain"].eq("baitbuster:youtube_human")]
if len(bb_brier) == 20 and (~bb_brier["balanced_brier_relative_change_estimable"].astype(bool)).all():
    axes[1,0].text(
        0.98, 0.97,
        "BaitBuster balanced-Brier endpoint:\nNE for all 20 model×seed settings",
        transform=axes[1,0].transAxes, ha="right", va="top", fontsize=7,
        bbox=dict(boxstyle="round,pad=0.25", facecolor="white", edgecolor="black", linewidth=0.6),
    )

plot_interval_points(
    axes[1,1], h2m.to_dict("records"), targets,
    "target_minus_source_gain", "h2_target_minus_source_gain_ci95", "h2_inferential_status",
    "D. H2 calibration-gain transport", "Target − source NLL gain"
)

fig.legend(handles=MODEL_LEGEND_HANDLES, loc="upper center", bbox_to_anchor=(0.5, 0.955),
           ncol=4, frameon=False, title="Model")
fig.suptitle("Figure 2. Cross-platform discrimination and probability reliability", y=0.995)
fig.text(
    0.5, 0.015,
    "BaitBuster overall H1 inference is non-estimable because AUROC and balanced-Brier bootstrap endpoints are incomplete under the locked procedure.",
    ha="center", va="bottom", fontsize=7,
)
fig.tight_layout(rect=[0, 0.055, 1, 0.93])
export_figure(fig, "FIG02_H1_H2", "Cross-platform discrimination and probability reliability", main=True)

register_source_data("FIG02_H1_H2", "A", auroc_plot,
                     ["Stage05/H1_RESULTS.json"], auroc_plot.columns,
                     "Direct calibrated AUROC point estimates; no pooling or new intervals.")
register_source_data("FIG02_H1_H2", "B_C", h1m,
                     ["Stage05/H1_RESULTS.json", "Stage05/STATISTICAL_SUMMARY.json"], h1m.columns,
                     "One-to-one merge by domain/model/seed of canonical H1 point estimates, endpoint-specific estimability flags, and canonical locked-bootstrap intervals/statuses.",
                     "nll_relative_change_estimable / balanced_brier_relative_change_estimable")
register_source_data("FIG02_H1_H2", "D", h2m,
                     ["Stage05/H2_RESULTS.json", "Stage05/STATISTICAL_SUMMARY.json"], h2m.columns,
                     "One-to-one merge by domain/model/seed of canonical H2 point estimates and canonical locked-bootstrap intervals/statuses.",
                     "h2_inferential_status")


PosixPath('/kaggle/working/08_manuscript_outputs/figure_source_data/FIG02_H1_H2_D_source.csv')

In [8]:
# 8. Figure 3 — Prespecified cue transport + estimability
h3df = H3_BC.copy()
h3df["platform"] = h3df["target"].str.split(":", n=1).str[1]
require(len(h3df) == 32, "Figure 3 source must preserve all 32 H3 cells")

fig, axes = plt.subplots(1, 2, figsize=(11.8, 7.6), sharey=True)
panels = [
    ("delta_rd", "delta_rd_ci95", "A. ΔRD (target − source cue risk-difference)"),
    ("ebm_delta_nll_gap_mean_5seeds", "ebm_delta_nll_gap_ci95", "B. ΔNLL-gap (EBM five-seed mean)"),
]
for ax, (value_col, ci_col, title) in zip(axes, panels):
    max_abs = max(1e-12, float(h3df[value_col].abs().max()))
    for yi, cue in enumerate(CUES):
        for xi, platform_name in enumerate(PLATFORMS):
            r = h3df[(h3df["cue"] == cue) & (h3df["platform"] == platform_name)].iloc[0]
            value = r[value_col]
            status = r["inferential_status"]
            if status != "ESTIMABLE":
                ax.plot(xi, yi, marker="x", linestyle="None", color="black", markersize=7)
                ax.text(xi, yi+0.20, "NE", ha="center", va="center", fontsize=6)
            else:
                marker = "o" if float(value) >= 0 else "s"
                size = 5 + 25 * min(1.0, abs(float(value)) / max_abs)
                ax.plot(xi, yi, marker=marker, linestyle="None", color="black",
                        markerfacecolor="white", markeredgecolor="black", markersize=size)
                lo, hi = ci_parts(r[ci_col])
                if np.isfinite(lo) and np.isfinite(hi):
                    ax.text(xi, yi+0.20, f"[{lo:.2g},{hi:.2g}]", ha="center", va="center", fontsize=5)
                ax.text(xi, yi-0.20, f"{float(value):.2g}", ha="center", va="center", fontsize=5)
    ax.set_xticks(range(4), [p.title() for p in PLATFORMS], rotation=20)
    ax.set_title(title, pad=8)
    ax.set_xlim(-0.5, 3.5)
    ax.set_ylim(len(CUES)-0.5, -0.5)
    ax.grid(True, color="0.85", linewidth=0.3)
axes[0].set_yticks(range(8), [cue_label(c) for c in CUES])
fig.legend(handles=[
    Line2D([0],[0], marker="o", linestyle="None", color="black", markerfacecolor="white", label="Positive point estimate"),
    Line2D([0],[0], marker="s", linestyle="None", color="black", markerfacecolor="white", label="Negative point estimate"),
    Line2D([0],[0], marker="x", linestyle="None", color="black", label="NE"),
], loc="upper center", bbox_to_anchor=(0.5, 0.945), ncol=3, frameon=False)
fig.suptitle(
    "Figure 3. Prespecified cue transport and estimability\n"
    f"Marker size = absolute point magnitude; NE = non-estimable under locked bootstrap; full 32-D global H3 status: {H3['global_banglaclick_outcome_wald']['status']}",
    y=0.995,
)
fig.tight_layout(rect=[0, 0, 1, 0.91])
export_figure(fig, "FIG03_H3_CUE_TRANSPORT", "Prespecified cue transport and estimability", main=True)
register_source_data("FIG03_H3_CUE_TRANSPORT", "A_B", h3df,
                     ["Stage05/H3_CUE_PLATFORM_CONTRASTS_CORRECTED.csv"],
                     h3df.columns,
                     "Exact fixed 32-cell table; marker shape encodes sign, marker size encodes absolute canonical point magnitude, NE cells retained explicitly; no imputation or reduced-dimensional analysis.",
                     "inferential_status")


PosixPath('/kaggle/working/08_manuscript_outputs/figure_source_data/FIG03_H3_CUE_TRANSPORT_A_B_source.csv')

In [9]:
# 9. Figure 4 — Fixed-budget operational degradation
h4b = pd.DataFrame(H4["bootstrap_rows"])
primary = h4b[np.isclose(h4b["budget"], PRIMARY_BUDGET)].copy()
require(len(primary) == 100, "Primary H4 inference table must have 100 rows")
bangla_primary = primary[primary["domain"].str.startswith("banglaclick:")].copy()
require(len(bangla_primary) == 80, "Primary BanglaClick H4 table must have 80 rows")

metric_specs = [
    ("target_minus_source_lift", "lift_ci95", "lift_estimability_status", "lift_p", "A. Lift: target − source"),
    ("target_minus_source_error_capture", "error_capture_ci95", "error_capture_estimability_status", "error_capture_p", "B. Error capture: target − source"),
    ("target_minus_source_aurc", "aurc_ci95", "aurc_estimability_status", "aurc_p", "C. AURC: target − source"),
]

fig, axes = plt.subplots(2, 3, figsize=(14.5, 8.5))
platform_domains = [f"banglaclick:{p}" for p in PLATFORMS]

# Primary 10% row: model = marker shape; canonical significance = fill; NE = x.
for ax, (field, ci_field, status_field, p_field, title) in zip(axes[0], metric_specs):
    for r in bangla_primary.to_dict("records"):
        model = r["model"]
        x = platform_domains.index(r["domain"]) + MODEL_OFFSETS[model]
        y = float(r[field])
        status = r[status_field]
        if status != "ESTIMABLE":
            ax.plot(x, y, marker="x", linestyle="None", color="black", markersize=4)
            continue
        lo, hi = ci_parts(r[ci_field])
        sig = r[p_field] is not None and np.isfinite(float(r[p_field])) and float(r[p_field]) < 0.05
        mfc = "black" if sig else "white"
        if np.isfinite(lo) and np.isfinite(hi):
            ax.errorbar(x, y, yerr=[[y-lo],[hi-y]], fmt=MODEL_MARKERS[model], color="black",
                        markerfacecolor=mfc, markeredgecolor="black", markersize=3.6,
                        capsize=1.5, linewidth=0.7)
        else:
            ax.plot(x, y, marker=MODEL_MARKERS[model], linestyle="None", color="black",
                    markerfacecolor=mfc, markeredgecolor="black", markersize=3.6)
    ax.axhline(0, color="black", linewidth=0.7)
    ax.set_xticks(range(4), [p.title() for p in PLATFORMS], rotation=20)
    ax.set_title(title)
    ax.set_ylabel("Effect")

# Budget sensitivity: budget = major x-position; platform = deterministic left-to-right offset;
# model marker semantics stay identical to the primary row. Canonical significance remains fill; NE remains x.
budget_order = [0.05, 0.10, 0.20]
for ax, (field, _, status_field, p_field, title) in zip(axes[1], metric_specs):
    for platform_name in PLATFORMS:
        dname = f"banglaclick:{platform_name}"
        sub = h4b[h4b["domain"].eq(dname)]
        for r in sub.to_dict("records"):
            model = r["model"]
            x = budget_order.index(float(r["budget"])) + PLATFORM_OFFSETS[platform_name]
            y = float(r[field])
            status = r[status_field]
            if status != "ESTIMABLE":
                ax.plot(x, y, marker="x", linestyle="None", color="black", markersize=3.5)
                continue
            sig = r[p_field] is not None and np.isfinite(float(r[p_field])) and float(r[p_field]) < 0.05
            ax.plot(x, y, marker=MODEL_MARKERS[model], linestyle="None", color="black",
                    markerfacecolor=("black" if sig else "white"), markeredgecolor="black", markersize=2.8)
    ax.axhline(0, color="black", linewidth=0.7)
    ax.set_xticks([0,1,2], ["5%", "10%", "20%"])
    ax.set_title(title.replace("A. ","").replace("B. ","").replace("C. ","") + " — budget sensitivity")
    ax.set_ylabel("Effect")
    ax.text(0.5, -0.20, "Within each budget, left→right: News · Facebook · YouTube · Twitter",
            transform=ax.transAxes, ha="center", va="top", fontsize=6.5)

fig.legend(handles=MODEL_LEGEND_HANDLES, loc="upper left", bbox_to_anchor=(0.12, 0.955),
           ncol=4, frameon=False, title="Model")
fig.legend(handles=STATUS_LEGEND_HANDLES, loc="upper right", bbox_to_anchor=(0.90, 0.955),
           ncol=3, frameon=False, title="Canonical status")
fig.suptitle("Figure 4. Fixed-budget operational degradation on BanglaClick\n10% is primary; 5% and 20% are prespecified sensitivity budgets", y=0.995)
fig.tight_layout(rect=[0, 0.06, 1, 0.90])
export_figure(fig, "FIG04_H4_FIXED_BUDGET", "Fixed-budget operational degradation", main=True)

register_source_data("FIG04_H4_FIXED_BUDGET", "primary_10pct", bangla_primary,
                     ["Stage06/H4_RESULTS.json"], bangla_primary.columns,
                     "Direct canonical Stage-06 10% bootstrap rows for the four BanglaClick target domains; model is marker shape, canonical significance is marker fill, NE is x; no pooling.",
                     "metric-specific *_estimability_status fields")
register_source_data("FIG04_H4_FIXED_BUDGET", "budget_sensitivity", h4b[h4b["domain"].str.startswith("banglaclick:")],
                     ["Stage06/H4_RESULTS.json"], h4b.columns,
                     "Direct canonical 5/10/20% Stage-06 bootstrap rows; budget is major x-position, platform is deterministic within-budget x-offset, model marker semantics are unchanged, and all model×seed settings are retained.")


PosixPath('/kaggle/working/08_manuscript_outputs/figure_source_data/FIG04_H4_FIXED_BUDGET_budget_sensitivity_source.csv')

In [10]:
# 10. Figure 5 — Exploratory H3→H4 bridge
pair_specs = [
    ("abs_delta_rd__vs__lift_deterioration", "abs_delta_rd", "lift_deterioration_source_minus_target", "|ΔRD|", "Lift deterioration (source − target)", "|ΔRD| → Lift deterioration"),
    ("abs_delta_rd__vs__error_capture_deterioration", "abs_delta_rd", "error_capture_deterioration_source_minus_target", "|ΔRD|", "Error-capture deterioration (source − target)", "|ΔRD| → Error-capture deterioration"),
    ("abs_delta_nll_gap__vs__lift_deterioration", "abs_delta_nll_gap", "lift_deterioration_source_minus_target", "|ΔNLL-gap|", "Lift deterioration (source − target)", "|ΔNLL-gap| → Lift deterioration"),
    ("abs_delta_nll_gap__vs__error_capture_deterioration", "abs_delta_nll_gap", "error_capture_deterioration_source_minus_target", "|ΔNLL-gap|", "Error-capture deterioration (source − target)", "|ΔNLL-gap| → Error-capture deterioration"),
]
fig, axes = plt.subplots(2, 2, figsize=(10.8, 8.4))
for ax, (pair, xcol, ycol, xlabel, ylabel, panel_title) in zip(axes.ravel(), pair_specs):
    for platform_name in PLATFORMS:
        sub = BRIDGE_CELLS[BRIDGE_CELLS["platform"].eq(platform_name)]
        ax.scatter(sub[xcol], sub[ycol], marker=PLATFORM_MARKERS[platform_name], s=24,
                   facecolors="white", edgecolors="black", linewidths=0.8, label=platform_name.title())
    point = BRIDGE["pairs"][pair]["point"]
    boot = BRIDGE["pairs"][pair]["bootstrap"]["platform_fe_slope"]
    ax.axhline(0, color="black", linewidth=0.6)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(panel_title, pad=7)
    ax.text(
        0.02, 0.98,
        f"Descriptive association only\nFE slope = {point['platform_fe_slope']:.4g}\n"
        "Inference: non-estimable under the locked joint bootstrap",
        transform=ax.transAxes, ha="left", va="top", fontsize=7,
        bbox=dict(boxstyle="round,pad=0.2", facecolor="white", edgecolor="0.4", linewidth=0.5),
    )
axes[0,0].legend(handles=PLATFORM_LEGEND_HANDLES, frameon=False, ncol=2, title="Platform")
fig.suptitle("Figure 5. Exploratory H3→H4 bridge\nDescriptive point associations only; no regression line, CI, or p-value is added", y=0.995)
fig.tight_layout(rect=[0, 0, 1, 0.94])
export_figure(fig, "FIG05_H3_H4_BRIDGE", "Exploratory H3→H4 bridge", main=True)

for pair, xcol, ycol, xlabel, ylabel, panel_title in pair_specs:
    df = BRIDGE_CELLS[["cue","platform",xcol,ycol]].copy()
    df["canonical_platform_fe_slope"] = BRIDGE["pairs"][pair]["point"]["platform_fe_slope"]
    df["inferential_status"] = BRIDGE["pairs"][pair]["bootstrap"]["platform_fe_slope"]["estimability_status"]
    register_source_data("FIG05_H3_H4_BRIDGE", pair, df,
                         ["Stage07/BRIDGE_32_CELL_POINT_TABLE.csv", "Stage07/H3_H4_BRIDGE_RESULTS.json"],
                         df.columns,
                         "Direct 32-cell descriptive scatter with canonical stored platform-FE slope annotation. No new line/intercept fit and no inferential interval.",
                         "inferential_status")

FIGURE_ISSUES.append({
    "figure_id": "FIG05_H3_H4_BRIDGE",
    "issue": "Canonical bridge artifacts store the platform-FE slope but no platform-specific intercepts. Therefore no regression line is drawn; only the stored descriptive slope is annotated.",
    "scientific_effect": "None; avoids fitting a new statistical model in the reproduction layer.",
})


In [11]:
# 11. Supplementary figures

# A. H3-P controlled cue-neutralization
h3p_df = pd.DataFrame(H3P["rows"])
h3p_figure_ids = [f"SUPP_H3P_{cue}" for cue in sorted(h3p_df["cue"].unique())]
for cue in sorted(h3p_df["cue"].unique()):
    subcue = h3p_df[h3p_df["cue"].eq(cue)].copy()
    fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharey=True)
    for ax, model in zip(axes, ["ebm", "banglabert"]):
        sm = subcue[subcue["model"].eq(model)].copy()
        targets = sorted(sm["target"].unique(), key=str)
        for ti, target in enumerate(targets):
            ss = sm[sm["target"].eq(target)].sort_values("seed")
            for si, r in enumerate(ss.to_dict("records")):
                y = ti + (si-2)*0.08
                val = r["target_minus_source_delta_z"]
                ci = r.get("ci95")
                status = r.get("inferential_status")
                if status == "ESTIMABLE" and val is not None:
                    lo, hi = ci_parts(ci)
                    if np.isfinite(lo) and np.isfinite(hi):
                        ax.errorbar(float(val), y, xerr=[[float(val)-lo],[hi-float(val)]], fmt="o",
                                    color="black", markerfacecolor="white", markersize=2.5, capsize=1)
                    else:
                        ax.plot(float(val), y, "o", color="black", markerfacecolor="white", markersize=2.5)
                else:
                    ax.text(0, y, "NE", fontsize=5)
        ax.axvline(0, color="black", linewidth=0.7)
        ax.set_yticks(range(len(targets)), [domain_label(t) for t in targets])
        ax.set_title(MODEL_LABELS.get(model, model))
        ax.set_xlabel("Δz = original − neutralized logit")
    fig.suptitle(f"Supplementary H3-P: {cue_label(cue)}\nControlled model-sensitivity evidence; non-causal")
    fig.tight_layout()
    export_figure(fig, f"SUPP_H3P_{cue}", f"H3-P {cue}", main=False)
register_source_data("SUPP_H3P_SHARED", "all", h3p_df, ["Stage05/H3_PERTURBATION_RESULTS.json"], h3p_df.columns,
                     "Shared canonical H3-P rows covering all three cue-neutralization supplementary figures; no pooling or new inference.",
                     "inferential_status", covers_figure_ids=h3p_figure_ids)

# B. BaitBuster H3 replication
bb = H3_BB.copy()
fig, axes = plt.subplots(1, 2, figsize=(11, 5.5), sharey=True)
for ax, (val, ci, title) in zip(axes, [
    ("delta_rd", "delta_rd_ci95", "ΔRD"),
    ("ebm_delta_nll_gap_mean_5seeds", "ebm_delta_nll_gap_ci95", "ΔNLL-gap"),
]):
    for i, r in enumerate(bb.sort_values("cue").to_dict("records")):
        if r["inferential_status"] == "ESTIMABLE":
            lo, hi = ci_parts(r[ci])
            if np.isfinite(lo) and np.isfinite(hi):
                ax.errorbar(float(r[val]), i, xerr=[[float(r[val])-lo],[hi-float(r[val])]], fmt="o",
                            color="black", markerfacecolor="white", markersize=3, capsize=1)
            else:
                ax.plot(float(r[val]), i, "o", color="black", markerfacecolor="white", markersize=3)
        else:
            ax.plot(float(r[val]), i, marker="x", linestyle="None", color="black")
            ax.text(float(r[val]), i+0.18, "NE", fontsize=6, ha="center")
    ax.axvline(0, color="black", linewidth=0.7)
    ax.set_title(title)
    ax.set_yticks(range(len(bb)), [cue_label(x) for x in bb.sort_values("cue")["cue"]])
fig.suptitle("Supplementary: BaitBuster independent-collection H3 replication")
fig.tight_layout()
export_figure(fig, "SUPP_BAITBUSTER_H3", "BaitBuster independent-collection H3 replication", main=False)
register_source_data("SUPP_BAITBUSTER_H3", "A_B", bb, ["Stage05/H3_BAITBUSTER_REPLICATION_CORRECTED.csv"], bb.columns,
                     "Direct canonical BaitBuster replication table; NE preserved.", "inferential_status")

# C. H3 sparsity / support diagnostics
sp = H3_SPARSE.copy()
count_cols = [c for c in sp.columns if "cue_present" in c and ("rows" in c or "n" == c)]
plot_col = count_cols[0] if count_cols else None
if plot_col is not None:
    pivot = sp.pivot(index="cue", columns="domain", values=plot_col).reindex(CUES)
    fig, ax = plt.subplots(figsize=(11, 6.5))
    for yi, cue in enumerate(pivot.index):
        for xi, domain in enumerate(pivot.columns):
            ax.text(xi, yi, f"{pivot.loc[cue,domain]:g}", ha="center", va="center", fontsize=6)
    ax.set_xticks(range(len(pivot.columns)), [domain_label(x) for x in pivot.columns], rotation=30, ha="right")
    ax.set_yticks(range(len(pivot.index)), [cue_label(x) for x in pivot.index])
    ax.set_title(f"Supplementary: H3 observed cue support ({plot_col})")
    ax.grid(True, color="0.85", linewidth=0.3)
    fig.tight_layout()
    export_figure(fig, "SUPP_H3_SPARSITY", "H3 sparsity and observed cue support", main=False)
else:
    OMISSIONS.append({"figure":"SUPP_H3_SPARSITY","reason":"Canonical sparsity artifact contains no recognized cue-present count column."})
register_source_data("SUPP_H3_SPARSITY", "support", sp, ["Stage05/H3_SPARSITY_REPORT.csv"], sp.columns,
                     "Direct canonical support diagnostics; no inferential recomputation.")

# D/E. H4 5% and 20%
for budget, tag in [(0.05, "05PCT"), (0.20, "20PCT")]:
    sub = pd.DataFrame(H4["bootstrap_rows"])
    sub = sub[np.isclose(sub["budget"], budget) & sub["domain"].str.startswith("banglaclick:")].copy()
    fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.2))
    for ax, (field, ci_field, status_field, p_field, title) in zip(axes, metric_specs):
        plot_interval_points(ax, sub.to_dict("records"), platform_domains, field, ci_field, status_field,
                             title, "Effect")
    fig.legend(handles=MODEL_LEGEND_HANDLES, loc="upper center", bbox_to_anchor=(0.5, 0.96), ncol=4, frameon=False, title="Model")
    fig.suptitle(f"Supplementary H4: {int(budget*100)}% review budget", y=0.995)
    fig.tight_layout(rect=[0,0,1,0.91])
    export_figure(fig, f"SUPP_H4_{tag}", f"H4 {int(budget*100)}% budget", main=False)
    register_source_data(f"SUPP_H4_{tag}", "A_C", sub, ["Stage06/H4_RESULTS.json"], sub.columns,
                         f"Direct canonical Stage-06 bootstrap rows filtered to budget={budget}; no pooling.")

# F. Risk-coverage curves — domain × model facets; seed is line style.
risk_domains = [
    "source_test",
    "banglaclick:news",
    "banglaclick:facebook",
    "banglaclick:youtube",
    "banglaclick:twitter",
    "baitbuster:youtube_human",
]
require(set(RISK_COVERAGE["domain"].unique()) == set(risk_domains), "Risk-coverage domain set changed")
require(set(RISK_COVERAGE["model"].unique()) == set(MODELS), "Risk-coverage model set changed")
require(set(int(x) for x in RISK_COVERAGE["seed"].unique()) == set(SEEDS), "Risk-coverage seed set changed")

fig, axes = plt.subplots(len(risk_domains), len(MODELS), figsize=(14.5, 15.0), sharex=False, sharey=False)
for ri, domain in enumerate(risk_domains):
    for ci, model in enumerate(MODELS):
        ax = axes[ri, ci]
        dm = RISK_COVERAGE[
            RISK_COVERAGE["domain"].eq(domain) & RISK_COVERAGE["model"].eq(model)
        ]
        for seed in SEEDS:
            g = dm[dm["seed"].eq(seed)].sort_values("coverage")
            require(len(g) > 0, f"Missing risk-coverage curve for {domain} / {model} / seed={seed}")
            ax.plot(g["coverage"], g["risk"], color="black", linewidth=0.65,
                    linestyle=SEED_LINESTYLES[seed], label=str(seed))
        if ri == 0:
            ax.set_title(MODEL_LABELS[model])
        if ci == 0:
            ax.set_ylabel(f"{domain_label(domain)}\nRisk")
        if ri == len(risk_domains)-1:
            ax.set_xlabel("Coverage")
seed_handles = [
    Line2D([0], [0], color="black", linestyle=SEED_LINESTYLES[seed], label=str(seed))
    for seed in SEEDS
]
fig.legend(handles=seed_handles, loc="upper center", bbox_to_anchor=(0.5, 0.985),
           ncol=5, frameon=False, title="Seed")
fig.suptitle(
    "Supplementary: canonical risk–coverage curves\n"
    "Domain = facet row; model = facet column; seed = line style; no averaging",
    y=1.005,
)
fig.tight_layout(rect=[0, 0, 1, 0.955])
export_figure(fig, "SUPP_RISK_COVERAGE", "Risk-coverage curves", main=False)
register_source_data("SUPP_RISK_COVERAGE", "all", RISK_COVERAGE, ["Stage06/RISK_COVERAGE_DATA.csv"], RISK_COVERAGE.columns,
                     "Direct canonical risk-coverage rows; each model×seed curve retained.")

# G. Expanded per-model/per-seed H1/H2
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
for ax, model in zip(axes.ravel(), MODELS):
    hm = h1m[h1m["model"].eq(model)]
    for seed in SEEDS:
        ss = hm[hm["seed"].eq(seed)].set_index("domain").reindex(targets)
        ax.plot(range(len(targets)), ss["nll_relative_change"], marker="o", color="black",
                markerfacecolor="white", linewidth=0.7, linestyle=SEED_LINESTYLES[seed], label=str(seed))
    ax.axhline(0, color="black", linewidth=0.6)
    ax.set_title(MODEL_LABELS[model])
    ax.set_xticks(range(len(targets)), [domain_label(x) for x in targets], rotation=30, ha="right")
    ax.set_ylabel("H1 relative NLL change")
axes[0,0].legend(frameon=False, fontsize=6, title="Seed")
fig.suptitle("Supplementary: H1 per-model/per-seed NLL transport")
fig.tight_layout()
export_figure(fig, "SUPP_H1_H2_MODEL_SEED", "Per-model/per-seed H1/H2", main=False)
register_source_data("SUPP_H1_H2_MODEL_SEED", "all", h1m,
                     ["Stage05/H1_RESULTS.json", "Stage05/STATISTICAL_SUMMARY.json"], h1m.columns,
                     "Exact canonical H1 model×seed rows used by the per-model/per-seed NLL transport figure; no pooling.",
                     "nll_relative_change_estimable")

# H. H4 model×seed primary results — domain facets, seed y-axis, consistent model markers.
h4_primary_all = pd.DataFrame(H4["bootstrap_rows"])
h4_primary_all = h4_primary_all[np.isclose(h4_primary_all["budget"], PRIMARY_BUDGET)].copy()
h4_domains = ["banglaclick:news", "banglaclick:facebook", "banglaclick:youtube", "banglaclick:twitter", "baitbuster:youtube_human"]
fig, axes = plt.subplots(len(h4_domains), 3, figsize=(14.5, 14.5), sharey=False)
for ri, domain in enumerate(h4_domains):
    dg = h4_primary_all[h4_primary_all["domain"].eq(domain)]
    for ci, (field, _, status_field, p_field, title) in enumerate(metric_specs):
        ax = axes[ri, ci]
        for r in dg.to_dict("records"):
            seed_y = SEEDS.index(int(r["seed"])) + 0.07 * (MODELS.index(r["model"]) - 1.5)
            yval = float(r[field])
            if r[status_field] != "ESTIMABLE":
                ax.plot(yval, seed_y, marker="x", linestyle="None", color="black", markersize=3.5)
            else:
                sig = r[p_field] is not None and np.isfinite(float(r[p_field])) and float(r[p_field]) < 0.05
                ax.plot(yval, seed_y, marker=MODEL_MARKERS[r["model"]], linestyle="None", color="black",
                        markerfacecolor=("black" if sig else "white"), markeredgecolor="black", markersize=3.2)
        ax.axvline(0, color="black", linewidth=0.6)
        ax.set_yticks(range(len(SEEDS)), [str(s) for s in SEEDS])
        if ci == 0:
            ax.set_ylabel(f"{domain_label(domain)}\nSeed")
        if ri == 0:
            ax.set_title(title.replace("A. ","").replace("B. ","").replace("C. ",""))
        if ri == len(h4_domains)-1:
            ax.set_xlabel("Target − source")
fig.legend(handles=MODEL_LEGEND_HANDLES, loc="upper left", bbox_to_anchor=(0.14, 0.995), ncol=4, frameon=False, title="Model")
fig.legend(handles=STATUS_LEGEND_HANDLES, loc="upper right", bbox_to_anchor=(0.90, 0.995), ncol=3, frameon=False, title="Canonical status")
fig.suptitle("Supplementary: H4 primary-budget model×seed results", y=1.015)
fig.tight_layout(rect=[0,0,1,0.97])
export_figure(fig, "SUPP_H4_MODEL_SEED", "H4 model×seed primary-budget results", main=False)
register_source_data("SUPP_H4_MODEL_SEED", "all", h4_primary_all,
                     ["Stage06/H4_RESULTS.json"], h4_primary_all.columns,
                     "Exact canonical 10% H4 model×seed rows. Domain is facet row, seed is the y-axis, model is marker shape, canonical significance is fill, and NE is x; no aggregation.",
                     "metric-specific *_estimability_status fields")

# I. LOO point robustness — descriptive only, inference NE
fig, axes = plt.subplots(2,2, figsize=(10.5,7.5))
loo_titles = {
    "abs_delta_rd__vs__lift_deterioration": "|ΔRD| → Lift deterioration",
    "abs_delta_rd__vs__error_capture_deterioration": "|ΔRD| → Error-capture deterioration",
    "abs_delta_nll_gap__vs__lift_deterioration": "|ΔNLL-gap| → Lift deterioration",
    "abs_delta_nll_gap__vs__error_capture_deterioration": "|ΔNLL-gap| → Error-capture deterioration",
}
for ax, pair in zip(axes.ravel(), sorted(LOO_POINTS["pair"].unique())):
    sub = LOO_POINTS[LOO_POINTS["pair"].eq(pair)].set_index("omitted_platform").reindex(PLATFORMS)
    ax.plot(range(4), sub["point_platform_fe_slope"], marker="o", linestyle="None",
            color="black", markerfacecolor="white")
    ax.axhline(0, color="black", linewidth=0.6)
    ax.set_xticks(range(4), [x.title() for x in PLATFORMS], rotation=25)
    ax.set_title(loo_titles[pair])
    ax.text(0.02,0.98,"Secondary robustness only\nInference: NE",transform=ax.transAxes,ha="left",va="top",fontsize=7)
fig.suptitle("Supplementary: leave-one-platform-out descriptive bridge slopes")
fig.tight_layout()
export_figure(fig, "SUPP_BRIDGE_LOO", "H3→H4 LOO robustness", main=False)
register_source_data("SUPP_BRIDGE_LOO", "all", LOO_POINTS, ["Stage07/LEAVE_ONE_PLATFORM_OUT_POINT_TABLE.csv","Stage07/LEAVE_ONE_PLATFORM_OUT_ROBUSTNESS.json"],
                     LOO_POINTS.columns, "Direct canonical LOO point slopes; no CI/p because every LOO inferential result is non-estimable.")

# J. Prevalence / structural sensitivity — domain facets; model marker and seed line style identify every canonical row.
sens_rows = []
for r in SENS["metric_rows"]:
    if r["domain"] == "source_test":
        continue
    base = r["calibrated"]
    prior = r.get("source_prior_standardized", {}).get("calibrated", {})
    struct = r.get("structural_composition_standardized", {}).get("calibrated", {})
    for adjustment, m in [("unadjusted", base), ("source_prior_standardized", prior), ("structural_composition_standardized", struct)]:
        if m:
            sens_rows.append({
                "domain":r["domain"],"model":r["model"],"seed":r["seed"],"adjustment":adjustment,
                "auroc":m.get("auroc"),"nll":m.get("nll"),"balanced_brier":m.get("balanced_brier")
            })
sens_df = pd.DataFrame(sens_rows)
adjustment_order = ["unadjusted", "source_prior_standardized", "structural_composition_standardized"]
adjustment_labels = ["Unadjusted", "Source-prior", "Structural"]
sens_domains = ["banglaclick:news", "banglaclick:facebook", "banglaclick:youtube", "banglaclick:twitter", "baitbuster:youtube_human"]
fig, axes = plt.subplots(len(sens_domains), 2, figsize=(12.5, 13.5), sharex=True)
for ri, domain in enumerate(sens_domains):
    dg = sens_df[sens_df["domain"].eq(domain)]
    for ci, field in enumerate(["auroc", "nll"]):
        ax = axes[ri, ci]
        for model in MODELS:
            for seed in SEEDS:
                ss = dg[(dg["model"].eq(model)) & (dg["seed"].eq(seed))].set_index("adjustment").reindex(adjustment_order)
                ax.plot(range(3), ss[field], marker=MODEL_MARKERS[model], color="black",
                        markerfacecolor="white", markeredgecolor="black", markersize=2.8,
                        linestyle=SEED_LINESTYLES[seed], linewidth=0.55, alpha=0.8)
        ax.set_xticks(range(3), adjustment_labels)
        if ci == 0:
            ax.set_ylabel(f"{domain_label(domain)}\nAUROC")
        else:
            ax.set_ylabel("NLL")
        if ri == 0:
            ax.set_title(field.upper())
seed_handles = [Line2D([0],[0], color="black", linestyle=SEED_LINESTYLES[s], label=str(s)) for s in SEEDS]
fig.legend(handles=MODEL_LEGEND_HANDLES, loc="upper left", bbox_to_anchor=(0.12, 0.995), ncol=4, frameon=False, title="Model")
fig.legend(handles=seed_handles, loc="upper right", bbox_to_anchor=(0.90, 0.995), ncol=5, frameon=False, title="Seed")
fig.suptitle("Supplementary: prevalence / structural-composition sensitivity\nEvery canonical domain×model×seed row is retained; no averaging", y=1.015)
fig.tight_layout(rect=[0,0,1,0.97])
export_figure(fig, "SUPP_SENSITIVITY", "Prevalence and structural sensitivity", main=False)
register_source_data("SUPP_SENSITIVITY", "A_B", sens_df, ["Stage05/SENSITIVITY_RESULTS.json"], sens_df.columns,
                     "Direct canonical unadjusted/source-prior/structural metric values. Domain is facet row, model is marker shape, seed is line style; no aggregation or new effect calculation.")


PosixPath('/kaggle/working/08_manuscript_outputs/figure_source_data/SUPP_SENSITIVITY_A_B_source.csv')

In [12]:
# 12. Main tables
study_map = pd.DataFrame([
    {"Hypothesis":"H1","Role":"Supporting","Question":"Discrimination / reliability decoupling under shift","Canonical artifact":"H1_RESULTS.json + STATISTICAL_SUMMARY.json"},
    {"Hypothesis":"H2","Role":"Supporting","Question":"Source-trained calibration-gain transport / attenuation","Canonical artifact":"H2_RESULTS.json + STATISTICAL_SUMMARY.json"},
    {"Hypothesis":"H3","Role":"Primary","Question":"Prespecified cue outcome/proper-loss transport","Canonical artifact":"H3_RESULTS.json + corrected H3 CSVs"},
    {"Hypothesis":"H3-P","Role":"Controlled corroboration","Question":"Cue-neutralization model sensitivity","Canonical artifact":"H3_PERTURBATION_RESULTS.json"},
    {"Hypothesis":"H4","Role":"Operational consequence","Question":"Fixed-budget review degradation","Canonical artifact":"H4_RESULTS.json"},
    {"Hypothesis":"H3→H4","Role":"Exploratory/non-causal","Question":"Descriptive bridge from cue instability to operational deterioration","Canonical artifact":"Stage-07 bridge artifacts"},
])
save_table(study_map, "TABLE01_STUDY_DESIGN_MAP", "Study design / hypothesis-to-analysis map", main=True)

h1_cons = pd.DataFrame(H1["domain_model_seed_consensus"]).assign(hypothesis="H1")
h2_cons = pd.DataFrame(H2["domain_model_seed_consensus"]).assign(hypothesis="H2")
h12_summary = pd.concat([h1_cons, h2_cons], ignore_index=True)
h12_display = h12_summary.copy()
h12_display["consensus_ge_4_of_5"] = h12_display["consensus_ge_4_of_5"].astype(object)
h12_display.loc[h12_display["nonestimable_seeds"].astype(int) > 0, "consensus_ge_4_of_5"] = "NE"
save_table(h12_display, "TABLE02_H1_H2_SUMMARY", "H1/H2 canonical model×domain seed-consensus summary", main=True)

h3_main_cols = [
    "cue","target","delta_rd","delta_rd_ci95","delta_rd_p",
    "ebm_delta_nll_gap_mean_5seeds","ebm_delta_nll_gap_ci95","ebm_delta_nll_gap_p",
    "inferential_status","valid_replicates","invalid_replicates",
]
h3_main_display = format_h3_display(H3_BC[h3_main_cols])
save_table(h3_main_display, "TABLE03_H3_CUE_TRANSPORT", "H3 cue transport summary with NE explicit", main=True)

# Deterministic summary of already-computed canonical H4 significance/statuses.
h4_primary = pd.DataFrame(H4["bootstrap_rows"])
h4_primary = h4_primary[np.isclose(h4_primary["budget"], PRIMARY_BUDGET)].copy()
summary_rows = []
for domain, g in h4_primary.groupby("domain", sort=True):
    for metric, effect, status, p in [
        ("lift","target_minus_source_lift","lift_estimability_status","lift_p"),
        ("error_capture","target_minus_source_error_capture","error_capture_estimability_status","error_capture_p"),
        ("aurc","target_minus_source_aurc","aurc_estimability_status","aurc_p"),
    ]:
        estimable = g[status].eq("ESTIMABLE")
        sig = estimable & g[p].notna() & (g[p].astype(float) < 0.05)
        if metric in {"lift","error_capture"}:
            degradation = g[effect].astype(float) < 0
        else:
            degradation = g[effect].astype(float) > 0
        summary_rows.append({
            "domain": domain,
            "metric": metric,
            "total_model_seed_settings": int(len(g)),
            "estimable_settings": int(estimable.sum()),
            "nonestimable_settings": int((~estimable).sum()),
            "canonical_p_lt_0_05_and_degradation_direction": int((sig & degradation).sum()),
        })
h4_summary = pd.DataFrame(summary_rows)
save_table(h4_summary, "TABLE04_H4_PRIMARY_10PCT_SUMMARY", "H4 primary 10% status/significance summary", main=True)


PosixPath('/kaggle/working/08_manuscript_outputs/tables/main/TABLE04_H4_PRIMARY_10PCT_SUMMARY.csv')

In [13]:
# 13. Supplementary tables
save_table(format_h3_display(H3_BC), "SUPP_TABLE01_FULL_H3_32_CELL", "Full H3 32-cell table", main=False)
save_table(H3_SPARSE, "SUPP_TABLE02_H3_SPARSITY", "H3 sparsity diagnostics", main=False)
save_table(pd.DataFrame(H3P["rows"]), "SUPP_TABLE03_H3P", "H3-P perturbation results", main=False)
save_table(format_h3_display(H3_BB), "SUPP_TABLE04_BAITBUSTER_H3", "BaitBuster H3 replication", main=False)
save_table(H4_FIXED, "SUPP_TABLE05_FULL_H4_BUDGET", "Full H4 fixed-budget point table", main=False)
save_table(format_h4_display(pd.DataFrame(H4["bootstrap_rows"])), "SUPP_TABLE06_H4_MODEL_SEED", "H4 model×seed bootstrap rows", main=False)
save_table(LOO_POINTS, "SUPP_TABLE07_LOO_POINT", "LOO bridge point results", main=False)

# Full statistical summary as deterministic row records, not recomputed statistics.
stat_rows = []
stat_rows.append({"source":"Stage05","key":"h1_overall_estimability","value_json":json.dumps(H1.get("estimability_policy"), sort_keys=True)})
stat_rows.append({"source":"Stage05","key":"statistical_summary","value_json":json.dumps(S5_STAT, sort_keys=True)})
stat_rows.append({"source":"Stage06","key":"final_statistical_summary","value_json":json.dumps(S6_STAT, sort_keys=True)})
stat_rows.append({"source":"Stage07","key":"bridge_results","value_json":json.dumps(BRIDGE, sort_keys=True)})
save_table(pd.DataFrame(stat_rows), "SUPP_TABLE08_FULL_STATISTICAL_SUMMARY", "Canonical statistical-summary references", main=False)

prov_table = pd.DataFrame([
    {"stage":"05","root":str(S5),"manifest_sha256":EXPECTED["stage05"]["manifest_sha256"],"checksum_sha256":EXPECTED["stage05"]["checksum_sha256"],"status_sha256":EXPECTED["stage05"]["status_sha256"]},
    {"stage":"06","root":str(S6),"manifest_sha256":EXPECTED["stage06"]["manifest_sha256"],"checksum_sha256":EXPECTED["stage06"]["checksum_sha256"],"status_sha256":EXPECTED["stage06"]["status_sha256"]},
    {"stage":"07","root":str(S7),"manifest_sha256":EXPECTED["stage07"]["manifest_sha256"],"checksum_sha256":EXPECTED["stage07"]["checksum_sha256"],"status_sha256":EXPECTED["stage07"]["status_sha256"]},
])
save_table(prov_table, "SUPP_TABLE09_PROVENANCE", "Provenance / reproducibility summary", main=False)


PosixPath('/kaggle/working/08_manuscript_outputs/tables/supplementary/SUPP_TABLE09_PROVENANCE.csv')

In [14]:
# 14. Figure source-data export
source_index = {
    "schema_version": 1,
    "reproduction_version": REPRODUCTION_VERSION,
    "source_files": SOURCE_REGISTRY,
    "rule": "Each generated figure must be explicitly covered by at least one registered exact deterministic source-data file; no source row is silently dropped for estimability.",
}
write_json(SRC_DATA / "FIGURE_SOURCE_DATA_INDEX.json", source_index)

# Internal validation: source CSV hashes equal the metadata hashes.
source_hash_ok = True
for meta in SOURCE_REGISTRY:
    p = SRC_DATA / meta["source_csv"]
    source_hash_ok = source_hash_ok and p.is_file() and sha256_file(p) == meta["source_csv_sha256"]
VALIDATION["figure_source_hash_match"] = bool(source_hash_ok)

# Mandatory coverage audit: every generated figure must map to explicit registered source data.
generated_figure_ids = {x["figure_id"] for x in FIGURE_REGISTRY}
covered_figure_ids = set()
for meta in SOURCE_REGISTRY:
    covered_figure_ids.update(meta.get("covers_figure_ids", [meta["figure_id"]]))
missing_source_coverage = sorted(generated_figure_ids - covered_figure_ids)
extra_source_coverage = sorted(covered_figure_ids - generated_figure_ids)
VALIDATION["every_figure_has_source_data"] = len(missing_source_coverage) == 0
write_json(OUT_ROOT / "FIGURE_SOURCE_COVERAGE.json", {
    "generated_figure_ids": sorted(generated_figure_ids),
    "covered_figure_ids": sorted(covered_figure_ids),
    "missing_source_coverage": missing_source_coverage,
    "extra_source_coverage": extra_source_coverage,
    "status": "PASS" if len(missing_source_coverage) == 0 else "FAIL",
})

expected_main = {
    "FIG01_STUDY_DESIGN", "FIG02_H1_H2", "FIG03_H3_CUE_TRANSPORT", "FIG04_H4_FIXED_BUDGET", "FIG05_H3_H4_BRIDGE"
}
expected_supp = {
    *{f"SUPP_H3P_{cue}" for cue in sorted(h3p_df["cue"].unique())},
    "SUPP_BAITBUSTER_H3", "SUPP_H3_SPARSITY", "SUPP_H4_05PCT", "SUPP_H4_20PCT",
    "SUPP_RISK_COVERAGE", "SUPP_H1_H2_MODEL_SEED", "SUPP_H4_MODEL_SEED", "SUPP_BRIDGE_LOO", "SUPP_SENSITIVITY",
}
actual_main = {x["figure_id"] for x in FIGURE_REGISTRY if x["main"]}
actual_supp = {x["figure_id"] for x in FIGURE_REGISTRY if not x["main"]}
VALIDATION["all_5_main_figures_registered"] = actual_main == expected_main
VALIDATION["all_supplementary_figures_registered"] = actual_supp == expected_supp
VALIDATION["h3_fixed_32_cells_preserved"] = len(H3_BC) == 32 and len(BRIDGE_CELLS) == 32
VALIDATION["h3_exactly_8_nonestimable_cells"] = int((H3_BC["inferential_status"] != "ESTIMABLE").sum()) == 8
VALIDATION["h3_global_nonestimable"] = H3["global_banglaclick_outcome_wald"]["status"] == "NON_ESTIMABLE"
VALIDATION["bridge_exploratory_noncausal"] = BRIDGE["exploratory_secondary"] is True and BRIDGE["causal"] is False
VALIDATION["bridge_all_primary_inferences_nonestimable"] = all(
    r["bootstrap"]["platform_fe_slope"]["estimability_status"] == "NON_ESTIMABLE_UNDER_LOCKED_JOINT_BOOTSTRAP"
    for r in BRIDGE["pairs"].values()
)
VALIDATION["h4_primary_budget_10pct"] = float(H4["primary_budget"]) == 0.10
VALIDATION["h4_secondary_budgets_5_20"] = sorted(SECONDARY_BUDGETS) == [0.05,0.20]
VALIDATION["no_random_plotting_jitter"] = True
VALIDATION["ne_ns_tokens_distinct"] = "NE" != "NS"
print("Figure source-data export and coverage validation complete.")


Figure source-data export and coverage validation complete.


In [15]:

# 15. Output manifest/checksum generation
# No scientific result is recomputed here. This only seals presentation artifacts.
generated_files = []
for p in sorted(OUT_ROOT.rglob("*"), key=lambda x: str(x)):
    if p.is_file() and p.name not in {"MANIFEST.json", "STAGE08_SHA256SUMS.txt", "FIGURE_REPRODUCTION_AUDIT.json", "STAGE08_STATUS.json"}:
        generated_files.append(p)

manifest = {
    "schema_version": 1,
    "stage_id": STAGE_ID,
    "reproduction_version": REPRODUCTION_VERSION,
    "protocol_version": PROTOCOL_VERSION,
    "scientific_role": "PURE_DETERMINISTIC_REPRODUCTION_NO_NEW_INFERENCE",
    "canonical_upstream": {
        "stage05_manifest_sha256": EXPECTED["stage05"]["manifest_sha256"],
        "stage06_manifest_sha256": EXPECTED["stage06"]["manifest_sha256"],
        "stage07_manifest_sha256": EXPECTED["stage07"]["manifest_sha256"],
    },
    "figures": FIGURE_REGISTRY,
    "tables": TABLE_REGISTRY,
    "figure_source_data_count": len(SOURCE_REGISTRY),
    "omissions": OMISSIONS,
    "artifacts": {
        str(p.relative_to(OUT_ROOT)).replace("\\","/"): sha256_file(p)
        for p in generated_files
    },
}
manifest["manifest_payload_sha256"] = hashlib.sha256(
    json.dumps(json_builtin(manifest), sort_keys=True, separators=(",", ":"), ensure_ascii=False).encode("utf-8")
).hexdigest()
write_json(OUT_ROOT / "MANIFEST.json", manifest)

checksum_lines = []
for p in sorted(OUT_ROOT.rglob("*"), key=lambda x: str(x)):
    if p.is_file() and p.name not in {"STAGE08_SHA256SUMS.txt", "STAGE08_STATUS.json", "FIGURE_REPRODUCTION_AUDIT.json"}:
        rel = str(p.relative_to(OUT_ROOT)).replace("\\","/")
        checksum_lines.append(f"{sha256_file(p)}  {rel}")
write_text(OUT_ROOT / "STAGE08_SHA256SUMS.txt", "\n".join(checksum_lines) + "\n")
print("Stage-08 manifest/checksum contract generated.")


Stage-08 manifest/checksum contract generated.


In [16]:
# 16. Final deterministic reproduction audit
# Re-check every manifest/checksum entry and source-data identity.
sums = parse_sha256sums(OUT_ROOT / "STAGE08_SHA256SUMS.txt")
checksum_ok = True
for rel, expected in sums.items():
    p = OUT_ROOT / rel
    checksum_ok = checksum_ok and p.is_file() and sha256_file(p) == expected
VALIDATION["stage08_checksum_contract"] = bool(checksum_ok)

ne_h3 = int((H3_BC["inferential_status"] != "ESTIMABLE").sum())
bridge_ne = {
    pair: result["bootstrap"]["platform_fe_slope"]["estimability_status"]
    for pair, result in BRIDGE["pairs"].items()
}

audit = {
    "schema_version": 1,
    "stage_id": STAGE_ID,
    "status": "PASS" if all(VALIDATION.values()) else "FAIL",
    "timestamp_utc": utc_now(),
    "scientific_role": "PURE_DETERMINISTIC_REPRODUCTION",
    "inputs_used": {
        "stage05_root": str(S5),
        "stage06_root": str(S6),
        "stage07_root": str(S7),
        "input_hashes_file": "INPUT_HASHES.json",
    },
    "figures_generated": FIGURE_REGISTRY,
    "tables_generated": TABLE_REGISTRY,
    "missing_inputs": [],
    "validation_results": VALIDATION,
    "ne_cell_counts": {
        "h3_banglaclick_ne_cells": ne_h3,
        "h3_total_cells": 32,
        "bridge_primary_inferential_status": bridge_ne,
    },
    "intentionally_omitted_or_modified_visual_elements": FIGURE_ISSUES + OMISSIONS,
    "determinism": {
        "plotting_randomness_used": False,
        "source_data_sorted_or_filtered_only_by_canonical_keys": True,
        "new_bootstrap": False,
        "new_hypothesis_test": False,
        "new_ci": False,
        "new_p_value": False,
        "new_model_or_calibration_fit": False,
        "figure_source_data_hashes_verified": VALIDATION.get("figure_source_hash_match", False),
        "every_generated_figure_has_source_data": VALIDATION.get("every_figure_has_source_data", False),
        "main_figure_registry_complete": VALIDATION.get("all_5_main_figures_registered", False),
        "supplementary_figure_registry_complete": VALIDATION.get("all_supplementary_figures_registered", False),
    },
}
write_json(OUT_ROOT / "FIGURE_REPRODUCTION_AUDIT.json", audit)
write_json(OUT_ROOT / "STAGE08_STATUS.json", {
    "status": audit["status"],
    "reason": "Canonical figures/tables reproduced without new scientific computation",
    "timestamp_utc": audit["timestamp_utc"],
    "output_root": str(OUT_ROOT),
})

# Refresh checksum contract to include final audit while excluding self-referential status/checksum.
checksum_lines = []
for p in sorted(OUT_ROOT.rglob("*"), key=lambda x: str(x)):
    if p.is_file() and p.name not in {"STAGE08_SHA256SUMS.txt", "STAGE08_STATUS.json"}:
        rel = str(p.relative_to(OUT_ROOT)).replace("\\","/")
        checksum_lines.append(f"{sha256_file(p)}  {rel}")
write_text(OUT_ROOT / "STAGE08_SHA256SUMS.txt", "\n".join(checksum_lines) + "\n")

if audit["status"] != "PASS":
    raise RuntimeError("Stage-08 deterministic reproduction audit failed")
print("STAGE 08 MANUSCRIPT REPRODUCTION — PASS")


STAGE 08 MANUSCRIPT REPRODUCTION — PASS
